# Idée 1

In [1]:
import ccxt, pandas as pd, time
from datetime import datetime, timezone

# ---------- Réglages ----------
EXCHANGE_ID = 'binance'
SYMBOL      = 'BTC/USDT'
TIMEFRAME   = '1d'
SINCE       = '2019-01-01'   # date de départ (modifiable)
CSV_PATH    = '../data/btc_spot_1d.csv'

# ---------- Exchange ----------
ex = getattr(ccxt, EXCHANGE_ID)({'enableRateLimit': True})
ex.load_markets()

# since -> ms epoch
since_ms = int(pd.Timestamp(SINCE, tz='UTC').timestamp()*1000)

# ---------- Téléchargement OHLCV en boucle ----------
rows = []
while True:
    batch = ex.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, since=since_ms, limit=1000)
    if not batch:
        break
    rows += batch
    since_ms = batch[-1][0] + 1
    time.sleep(ex.rateLimit/1000)

# ---------- DataFrame ----------
cols = ['timestamp','open','high','low','close','volume']
df = pd.DataFrame(rows, columns=cols)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)

# ---------- Variables quantitatives ----------
df['ret']      = df['close'].pct_change()                         # continue
df['range']    = df['high'] - df['low']                           # continue
df['day']      = df['date'].dt.day                                # discrète
df['month']    = df['date'].dt.month                              # discrète
df['weekday']  = df['date'].dt.weekday                            # discrète (0=lundi)
df['vol_10d']  = df['ret'].rolling(10).std() * (252**0.5)         # vol annualisée approx.

# ---------- Variables qualitatives ----------
# Nominale: tendance (hausse/baisse) par rapport à MM20
mm20 = df['close'].rolling(20).mean()
df['trend'] = (df['close'] > mm20).map({True: 'Hausse', False: 'Baisse'})

# Ordinale: niveau de volatilité (seuils ajustables)
def vol_bucket(x):
    if pd.isna(x): return pd.NA
    if x < 0.4:    return 'Faible'
    if x < 0.8:    return 'Moyenne'
    return 'Élevée'
df['vol_level'] = df['vol_10d'].apply(vol_bucket)

# Nominale: saison (libellés lisibles)
mois = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Juin',7:'Juil',8:'Août',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}
df['month_name'] = df['month'].map(mois)

# ---------- Nettoyage & export ----------
df = df.dropna(subset=['open','high','low','close','volume'])
df.to_csv(CSV_PATH, index=False)
print(f"[OK] Écrit : {CSV_PATH}  ({len(df)} lignes)")


[OK] Écrit : ../data/btc_spot_1d.csv  (2497 lignes)


In [2]:
data = pd.read_csv(CSV_PATH)
data.to_excel('../data/btc_spot_1d.xlsx', index=False)
data

,timestamp,open,high,low,close,volume,date,ret,range,day,month,weekday,vol_10d,trend,vol_level,month_name
0,1546300800000,3701.23,3810.16,3642.00,3797.14,23741.687033,2019-01-01 00:00:00+00:00,NaN,168.16,1,1,1,NaN,Baisse,NaN,Jan
1,1546387200000,3796.45,3882.14,3750.45,3858.56,35156.463369,2019-01-02 00:00:00+00:00,0.016175,131.69,2,1,2,NaN,Baisse,NaN,Jan
2,1546473600000,3857.57,3862.74,3730.00,3766.78,29406.948359,2019-01-03 00:00:00+00:00,-0.023786,132.74,3,1,3,NaN,Baisse,NaN,Jan
3,1546560000000,3767.20,3823.64,3703.57,3792.01,29519.554671,2019-01-04 00:00:00+00:00,0.006698,120.07,4,1,4,NaN,Baisse,NaN,Jan
4,1546646400000,3790.09,3840.99,3751.00,3770.96,30490.667751,2019-01-05 00:00:00+00:00,-0.005551,89.99,5,1,5,NaN,Baisse,NaN,Jan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2492,1761609600000,114107.65,116086.00,112211.00,112898.45,15523.422570,2025-10-28 00:00:00+00:00,-0.010597,3875.00,28,10,1,0.243469,Hausse,Faible,Oct
2493,1761696000000,112898.44,113643.73,109200.00,110021.29,21079.713760,2025-10-29 00:00:00+00:00,-0.025484,4443.73,29,10,2,0.282192,Baisse,Faible,Oct
2494,1761782400000,110021.30,111592.00,106304.34,108322.88,25988.828380,2025-10-30 00:00:00+00:00,-0.015437,5287.66,30,10,3,0.278209,Baisse,Faible,Oct
2495,1761868800000,108322.87,111190.00,108275.28,109608.01,21518.204390,2025-10-31 00:00:00+00:00,0.011864,2914.72,31,10,4,0.265318,Baisse,Faible,Oct


In [3]:
import ccxt, pandas as pd, time
from datetime import datetime, timezone

# ---------- Réglages ----------
EXCHANGE_ID = 'binance'
SYMBOL      = 'BTC/USDT'
TIMEFRAME   = '1h'
SINCE       = '2019-01-01'   # date de départ (modifiable)
CSV_PATH    = '../data/btc_spot_1h.csv'

# ---------- Exchange ----------
ex = getattr(ccxt, EXCHANGE_ID)({'enableRateLimit': True})
ex.load_markets()

# since -> ms epoch
since_ms = int(pd.Timestamp(SINCE, tz='UTC').timestamp()*1000)

# ---------- Téléchargement OHLCV en boucle ----------
rows = []
while True:
    batch = ex.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, since=since_ms, limit=1000)
    if not batch:
        break
    rows += batch
    since_ms = batch[-1][0] + 1
    time.sleep(ex.rateLimit/1000)

# ---------- DataFrame ----------
cols = ['timestamp','open','high','low','close','volume']
df = pd.DataFrame(rows, columns=cols)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)

# ---------- Variables quantitatives ----------
df['ret']      = df['close'].pct_change()                         # continue
df['range']    = df['high'] - df['low']                           # continue
df['day']      = df['date'].dt.day                                # discrète
df['month']    = df['date'].dt.month                              # discrète
df['weekday']  = df['date'].dt.weekday                            # discrète (0=lundi)
df['vol_10d']  = df['ret'].rolling(10).std() * (252**0.5)         # vol annualisée approx.

# ---------- Variables qualitatives ----------
# Nominale: tendance (hausse/baisse) par rapport à MM20
mm20 = df['close'].rolling(20).mean()
df['trend'] = (df['close'] > mm20).map({True: 'Hausse', False: 'Baisse'})

# Ordinale: niveau de volatilité (seuils ajustables)
def vol_bucket(x):
    if pd.isna(x): return pd.NA
    if x < 0.4:    return 'Faible'
    if x < 0.8:    return 'Moyenne'
    return 'Élevée'
df['vol_level'] = df['vol_10d'].apply(vol_bucket)

# Nominale: saison (libellés lisibles)
mois = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Juin',7:'Juil',8:'Août',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}
df['month_name'] = df['month'].map(mois)

# ---------- Nettoyage & export ----------
df = df.dropna(subset=['open','high','low','close','volume'])
df.to_csv(CSV_PATH, index=False)
print(f"[OK] Écrit : {CSV_PATH}  ({len(df)} lignes)")


[OK] Écrit : ../data/btc_spot_1h.csv  (59863 lignes)


In [4]:
data = pd.read_csv(CSV_PATH)
data.to_excel('../data/btc_spot_1h.xlsx', index=False)
data

,timestamp,open,high,low,close,volume,date,ret,range,day,month,weekday,vol_10d,trend,vol_level,month_name
0,1546300800000,3701.23,3713.00,3689.88,3700.31,686.367420,2019-01-01 00:00:00+00:00,NaN,23.12,1,1,1,NaN,Baisse,NaN,Jan
1,1546304400000,3700.20,3702.73,3684.22,3689.69,613.539115,2019-01-01 01:00:00+00:00,-0.002870,18.51,1,1,1,NaN,Baisse,NaN,Jan
2,1546308000000,3689.67,3695.95,3675.04,3690.00,895.302181,2019-01-01 02:00:00+00:00,0.000084,20.91,1,1,1,NaN,Baisse,NaN,Jan
3,1546311600000,3690.00,3699.77,3685.78,3693.13,796.714818,2019-01-01 03:00:00+00:00,0.000848,13.99,1,1,1,NaN,Baisse,NaN,Jan
4,1546315200000,3692.32,3720.00,3685.94,3692.71,1317.452909,2019-01-01 04:00:00+00:00,-0.000114,34.06,1,1,1,NaN,Baisse,NaN,Jan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59858,1762005600000,109946.26,109993.98,109728.94,109933.85,343.690110,2025-11-01 14:00:00+00:00,-0.000113,265.04,1,11,5,0.020098,Hausse,Faible,Nov
59859,1762009200000,109933.86,110348.00,109910.22,110310.65,619.565240,2025-11-01 15:00:00+00:00,0.003428,437.78,1,11,5,0.026517,Hausse,Faible,Nov
59860,1762012800000,110310.65,110529.99,110208.49,110464.33,670.420390,2025-11-01 16:00:00+00:00,0.001393,321.50,1,11,5,0.025935,Hausse,Faible,Nov
59861,1762016400000,110464.33,110495.04,110192.00,110208.97,297.515560,2025-11-01 17:00:00+00:00,-0.002312,303.04,1,11,5,0.029266,Hausse,Faible,Nov


# Idée 2

In [5]:
# build_dataset.py
# -*- coding: utf-8 -*-
"""
Construit un dataset financier multi-actifs, prêt pour :
- Statistiques univariées/bivariées
- Corrélations et régressions
- ACP (variables quantitatives standardisées)
- AFC (croisements de variables qualitatives dérivées)

Dépendances : ccxt, yfinance, pandas, numpy, pyarrow
"""

import time
import math
import numpy as np
import pandas as pd
import ccxt
import yfinance as yf
from datetime import datetime, timezone

# ----------------- Réglages -----------------
START_DATE = "2019-01-01"
EXCHANGE_ID = "binance"
CRYPTO_PAIRS = ["BTC/USDT", "ETH/USDT"]       # extensible
YF_TICKERS  = {"SP500": "^GSPC", "GOLD": "GC=F"}  # Indices/Commodities
TIMEFRAME   = "1d"
OUT_PATH    = "../data/multi_asset_dataset.parquet"  # Parquet recommandé

# ----------------- Utilitaires -----------------
def fetch_ohlcv_ccxt(exchange, symbol, timeframe, since_iso):
    """Télécharge l'historique OHLCV complet via pagination."""
    since_ms = int(pd.Timestamp(since_iso, tz='UTC').timestamp() * 1000)
    all_rows = []
    while True:
        try:
            batch = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since_ms, limit=1000)
        except Exception:
            time.sleep(exchange.rateLimit / 1000 * 2)
            continue
        if not batch:
            break
        all_rows += batch
        since_ms = batch[-1][0] + 1
        time.sleep(exchange.rateLimit / 1000)
        # Stop condition to avoid infinite loop if exchange loops same last candle
        if len(batch) < 1000:
            # may still have more; continue until empty
            pass
        # safety for rare duplicates
        if len(all_rows) >= 1 and len(all_rows) % 100000 == 0:
            print(f"{symbol}: {len(all_rows)} lignes récupérées...")
    df = pd.DataFrame(all_rows, columns=["timestamp","open","high","low","close","volume"])
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert(None)
    df["symbol"] = symbol.split("/")[0]
    return df[["date","symbol","open","high","low","close","volume"]]

def pct_change(s):
    return s.pct_change(fill_method=None)

def rolling_vol(ret, window):
    return ret.rolling(window).std() * np.sqrt(252)

def quantile_buckets(s, qs=(0, .33, .66, 1), labels=None):
    q = s.quantile(qs)
    q = q.drop_duplicates()
    if labels is None:
        labels = [f"Q{i+1}" for i in range(len(q)-1)]
    return pd.cut(s, bins=q, include_lowest=True, labels=labels)

# ----------------- Collecte -----------------
print("[+] Téléchargement crypto via CCXT…")
ex = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
ex.load_markets()
dfs = []
for pair in CRYPTO_PAIRS:
    if not ex.has.get("fetchOHLCV"):
        raise RuntimeError(f"L'exchange {EXCHANGE_ID} ne supporte pas fetchOHLCV.")
    df_pair = fetch_ohlcv_ccxt(ex, pair, TIMEFRAME, START_DATE)
    dfs.append(df_pair)

crypto = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
if crypto.empty:
    raise RuntimeError("Aucune donnée crypto récupérée.")

# pivot prix/volume
crypto_p = crypto.pivot(index="date", columns="symbol", values="close").rename(
    columns={s.split("/")[0] if "/" in s else s: s for s in [p.split('/')[0] for p in CRYPTO_PAIRS]}
)
crypto_v = crypto.pivot(index="date", columns="symbol", values="volume")
crypto_p.columns = [f"close_{c}" for c in crypto_p.columns]
crypto_v.columns = [f"vol_{c}" for c in crypto_v.columns]

print("[+] Téléchargement marchés traditionnels via yfinance…")
others = []
for name, ticker in YF_TICKERS.items():
    yfd = yf.download(ticker, start=START_DATE, progress=False, auto_adjust=False)
    if yfd.empty:
        continue
    
    # Aplatir les colonnes multi-niveaux si nécessaire
    if isinstance(yfd.columns, pd.MultiIndex):
        yfd.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in yfd.columns.values]
    
    # Trouver la colonne "Close" (peut être "Close" ou "Close_TICKER")
    close_col = None
    for col in yfd.columns:
        if 'Close' in col or col == 'Close':
            close_col = col
            break
    
    if close_col is None:
        print(f"[!] Pas de colonne 'Close' trouvée pour {name}, colonnes disponibles: {yfd.columns.tolist()}")
        continue
    
    # Renommer et sélectionner la colonne
    yfd = yfd[[close_col]].rename(columns={close_col: f"close_{name}"})
    yfd.index = pd.to_datetime(yfd.index)
    others.append(yfd)

macro = pd.concat(others, axis=1) if others else pd.DataFrame()

# ----------------- Fusion et features -----------------
tab = crypto_p.join(crypto_v, how="outer")
if not macro.empty:
    tab = tab.join(macro, how="outer")
tab = tab.sort_index()
tab = tab.loc[~tab.index.duplicated(keep="first")]
tab.index.name = "date"

# Rendements (journaliers)
for col in [c for c in tab.columns if c.startswith("close_")]:
    asset = col.replace("close_","")
    tab[f"ret_{asset}"] = tab[col].pct_change(fill_method=None)

# Volatilités glissantes (30 jours ~ 1 mois)
for asset in set([c.replace("close_","") for c in tab.columns if c.startswith("close_")]):
    tab[f"vol30_{asset}"] = rolling_vol(tab[f"ret_{asset}"], 30)

# Corrélations roulantes (BTC vs autres, fenêtre 30j)
base = "BTC" if "ret_BTC" in tab.columns else None
if base:
    for asset in [a for a in set([c.replace("ret_","") for c in tab.columns if c.startswith("ret_")]) if a != base]:
        tab[f"corr30_{base}_{asset}"] = tab["ret_"+base].rolling(30).corr(tab["ret_"+asset])

# Variables calendaires & qualitatives (pour quali↔quali / quali↔quanti)
tab["day"] = tab.index.day
tab["month"] = tab.index.month
tab["weekday"] = tab.index.weekday
mois = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Juin',7:'Juil',8:'Août',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}
tab["month_name"] = tab["month"].map(mois)

# Choisir un actif pivot pour générer des catégories (ex: BTC)
pivot_close = "close_BTC" if "close_BTC" in tab.columns else [c for c in tab.columns if c.startswith("close_")][0]
pivot_ret   = "ret_BTC"   if "ret_BTC"   in tab.columns else [c for c in tab.columns if c.startswith("ret_")][0]
pivot_vol30 = "vol30_BTC" if "vol30_BTC" in tab.columns else [c for c in tab.columns if c.startswith("vol30_")][0]

# Tendance (nominale) : au-dessus/dessous de la MM20
mm20 = tab[pivot_close].rolling(20).mean()
tab["trend"] = np.where(tab[pivot_close] > mm20, "Hausse", "Baisse")

# Niveau de volatilité (ordinale) via quantiles sur vol30
tab["vol_level"] = quantile_buckets(tab[pivot_vol30], qs=(0,.33,.66,1), labels=["Faible","Moyenne","Élevée"])

# Buckets de rendement quotidien (ordinale)
tab["ret_bucket"] = quantile_buckets(tab[pivot_ret].clip(-0.2,0.2), qs=(0,.25,.5,.75,1), labels=["T1","T2","T3","T4"])

# Nettoyage final
tab = tab.dropna(how="all")
tab = tab.reset_index()

# Colonnes utiles en premier
first_cols = ["date", pivot_close, pivot_ret, pivot_vol30, "trend", "vol_level", "ret_bucket", "month_name", "weekday"]
other_cols = [c for c in tab.columns if c not in first_cols]
tab = tab[first_cols + other_cols]

# Export Parquet + CSV + XLSX
csv_path = OUT_PATH.replace(".parquet", ".csv")
xlsx_path = OUT_PATH.replace(".parquet", ".xlsx")
tab.to_parquet(OUT_PATH, index=False)
tab.to_csv(csv_path, index=False, encoding="utf-8")
tab.to_excel(xlsx_path, index=False)
print(f"[OK] Dataset écrit : {OUT_PATH}, {csv_path} et {xlsx_path} (lignes={len(tab)})")


[+] Téléchargement crypto via CCXT…
[+] Téléchargement marchés traditionnels via yfinance…
[OK] Dataset écrit : ../data/multi_asset_dataset.parquet, ../data/multi_asset_dataset.csv et ../data/multi_asset_dataset.xlsx (lignes=2497)


Ajout du fear index 

In [1]:
"""
============================================================================
INTÉGRATION DE L'INDICE FEAR & GREED AU DATASET BITCOIN
============================================================================
Ce script récupère l'indice Fear & Greed d'Alternative.me et l'intègre
au dataset existant pour enrichir l'analyse avec une variable qualitative
supplémentaire représentant le sentiment du marché.

Auteur: Simon HAMELIN
Date: 2025
============================================================================
"""

import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. FONCTION DE RÉCUPÉRATION DE L'INDICE FEAR & GREED
# ============================================================================

def fetch_fear_greed(start_date="2019-01-01", end_date=None):
    """
    Récupère l'indice Fear & Greed depuis l'API Alternative.me.
    
    L'indice varie de 0 (Extreme Fear) à 100 (Extreme Greed) et mesure
    le sentiment général du marché crypto basé sur plusieurs facteurs:
    - Volatilité (25%)
    - Momentum/Volume (25%)
    - Social Media (15%)
    - Dominance Bitcoin (10%)
    - Trends Google (10%)
    - Surveys (15%)
    
    Args:
        start_date (str): Date de début au format 'YYYY-MM-DD'
        end_date (str): Date de fin au format 'YYYY-MM-DD' (None = aujourd'hui)
    
    Returns:
        pd.DataFrame: DataFrame avec colonnes ['date', 'fgi_score', 'fgi_classification']
                      indexé par date
    """
    print("\n" + "="*70)
    print("RÉCUPÉRATION DE L'INDICE FEAR & GREED")
    print("="*70)
    
    # URL de l'API (limit=0 récupère toutes les données disponibles)
    url = "https://api.alternative.me/fng/?limit=0&format=json"
    
    try:
        print(f"[*] Requête vers: {url}")
        r = requests.get(url, timeout=15)
        r.raise_for_status()  # Lève une exception si status != 200
        
        # Extraction des données JSON
        data = r.json()["data"]
        print(f"[✓] {len(data)} enregistrements récupérés")
        
        # Conversion en DataFrame
        fgi = pd.DataFrame(data)
        
        # Nettoyage et transformation des données
        # 1. Convertir le score (string) en integer
        fgi["fgi_score"] = fgi["value"].astype(int)
        
        # 2. Conserver la classification textuelle
        fgi["fgi_classification"] = fgi["value_classification"]
        
        # 3. Convertir le timestamp Unix en datetime
        #    timestamp est en secondes depuis epoch (1970-01-01)
        fgi["date"] = pd.to_datetime(
            fgi["timestamp"].astype(int), 
            unit="s",           # timestamp en secondes
            utc=True            # timezone UTC
        ).dt.tz_convert(None)   # Retirer timezone
        
        # 4. Normaliser au jour (supprimer heures/minutes/secondes)
        fgi["date"] = fgi["date"].dt.normalize()
        
        # 5. Sélectionner et nettoyer les colonnes
        fgi = fgi[["date", "fgi_score", "fgi_classification"]]
        fgi = fgi.drop_duplicates("date").sort_values("date")
        
        print(f"[*] Période disponible: {fgi['date'].min()} à {fgi['date'].max()}")
        
        # 6. Filtrer selon les dates demandées
        start = pd.to_datetime(start_date)
        fgi = fgi[fgi["date"] >= start]
        
        if end_date:
            end = pd.to_datetime(end_date)
            fgi = fgi[fgi["date"] <= end]
        
        print(f"[*] Période filtrée: {fgi['date'].min()} à {fgi['date'].max()}")
        print(f"[✓] {len(fgi)} jours après filtrage")
        
        # 7. Définir l'index et créer une série temporelle continue
        fgi = fgi.set_index("date")
        
        # Forward-fill pour remplir les jours manquants (week-ends, etc.)
        # Cela propage la dernière valeur connue
        fgi = fgi.asfreq("D").ffill()
        
        print(f"[✓] {len(fgi)} jours après interpolation (forward-fill)")
        
        # Statistiques descriptives
        print("\n" + "-"*70)
        print("STATISTIQUES FEAR & GREED")
        print("-"*70)
        print(f"Score moyen:    {fgi['fgi_score'].mean():.2f}")
        print(f"Score médian:   {fgi['fgi_score'].median():.2f}")
        print(f"Score min:      {fgi['fgi_score'].min()}")
        print(f"Score max:      {fgi['fgi_score'].max()}")
        print(f"Écart-type:     {fgi['fgi_score'].std():.2f}")
        
        # Distribution par classification
        print("\n" + "-"*70)
        print("DISTRIBUTION PAR CATÉGORIE")
        print("-"*70)
        counts = fgi["fgi_classification"].value_counts()
        for cat, count in counts.items():
            pct = (count / len(fgi)) * 100
            print(f"{cat:20s}: {count:5d} jours ({pct:5.1f}%)")
        
        return fgi
        
    except requests.exceptions.RequestException as e:
        print(f"[!] ERREUR lors de la requête HTTP: {e}")
        return pd.DataFrame(columns=["fgi_score", "fgi_classification"])
    
    except Exception as e:
        print(f"[!] ERREUR inattendue: {e}")
        return pd.DataFrame(columns=["fgi_score", "fgi_classification"])


# ============================================================================
# 2. FONCTION DE CRÉATION DE VARIABLES DÉRIVÉES
# ============================================================================

def create_fgi_features(df):
    """
    Crée des variables dérivées à partir du score Fear & Greed.
    
    Args:
        df (pd.DataFrame): DataFrame avec colonne 'fgi_score'
    
    Returns:
        pd.DataFrame: DataFrame enrichi avec nouvelles variables
    """
    print("\n" + "="*70)
    print("CRÉATION DE VARIABLES DÉRIVÉES")
    print("="*70)
    
    df = df.copy()
    
    # 1. Catégories selon échelle standard
    #    0-25:   Extreme Fear
    #    26-45:  Fear
    #    46-55:  Neutral
    #    56-75:  Greed
    #    76-100: Extreme Greed
    def categorize_fgi(score):
        if score <= 25:
            return "Extreme Fear"
        elif score <= 45:
            return "Fear"
        elif score <= 55:
            return "Neutral"
        elif score <= 75:
            return "Greed"
        else:
            return "Extreme Greed"
    
    df["fgi_category"] = df["fgi_score"].apply(categorize_fgi)
    print("[✓] Catégories créées (5 niveaux)")
    
    # 2. Catégorie simplifiée (3 niveaux)
    def categorize_fgi_simple(score):
        if score < 40:
            return "Fear"
        elif score < 60:
            return "Neutral"
        else:
            return "Greed"
    
    df["fgi_simple"] = df["fgi_score"].apply(categorize_fgi_simple)
    print("[✓] Catégories simplifiées créées (3 niveaux)")
    
    # 3. Variation du FGI (changement jour à jour)
    df["fgi_change"] = df["fgi_score"].diff()
    print("[✓] Variation quotidienne calculée")
    
    # 4. Variation en pourcentage
    df["fgi_pct_change"] = df["fgi_score"].pct_change() * 100
    print("[✓] Variation % calculée")
    
    # 5. Moyenne mobile 7 jours
    df["fgi_ma7"] = df["fgi_score"].rolling(window=7, min_periods=1).mean()
    print("[✓] Moyenne mobile 7j calculée")
    
    # 6. Moyenne mobile 30 jours
    df["fgi_ma30"] = df["fgi_score"].rolling(window=30, min_periods=1).mean()
    print("[✓] Moyenne mobile 30j calculée")
    
    # 7. Écart à la moyenne mobile 30j (signal)
    df["fgi_signal"] = df["fgi_score"] - df["fgi_ma30"]
    print("[✓] Signal (écart MA30) calculé")
    
    # 8. Volatilité du FGI (écart-type mobile 30j)
    df["fgi_volatility"] = df["fgi_score"].rolling(window=30, min_periods=1).std()
    print("[✓] Volatilité 30j calculée")
    
    # 9. Indicateur de changement de sentiment
    #    +1 si passage de Fear à Greed, -1 si Greed à Fear, 0 sinon
    df["fgi_regime_change"] = 0
    for i in range(1, len(df)):
        prev_simple = df["fgi_simple"].iloc[i-1]
        curr_simple = df["fgi_simple"].iloc[i]
        
        if prev_simple in ["Fear", "Neutral"] and curr_simple == "Greed":
            df.loc[df.index[i], "fgi_regime_change"] = 1
        elif prev_simple in ["Greed", "Neutral"] and curr_simple == "Fear":
            df.loc[df.index[i], "fgi_regime_change"] = -1
    
    print("[✓] Changements de régime détectés")
    
    return df


# ============================================================================
# 3. CHARGEMENT DES TAUX D'INTÉRÊT DE LA FED
# ============================================================================

def load_fed_interest_rates(csv_path="../data/taux de la fed.csv", start_date=None, end_date=None):
    """
    Charge les taux d'intérêt (Fed) depuis un fichier CSV, convertit en fréquence
    quotidienne avec forward-fill et applique les filtres de dates fournis.

    Args:
        csv_path (str): Chemin vers le CSV des taux.
        start_date (str|pd.Timestamp|None): Date minimale à conserver.
        end_date (str|pd.Timestamp|None): Date maximale à conserver.

    Returns:
        pd.DataFrame: DataFrame indexé par date avec la colonne 'fed_real_rate'.
    """
    print("\n" + "="*70)
    print("CHARGEMENT DES TAUX D'INTÉRÊT (FED)")
    print("="*70)

    try:
        fed = pd.read_csv(csv_path, parse_dates=["observation_date"])
        print(f"[✓] Fichier Fed chargé: {len(fed)} lignes brutes")
    except FileNotFoundError:
        print(f"[!] Fichier introuvable: {csv_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"[!] ERREUR lors du chargement du fichier Fed: {e}")
        return pd.DataFrame()

    fed = fed.rename(columns={"observation_date": "date", "REAINTRATREARAT10Y": "fed_real_rate"})
    fed = fed.set_index("date").sort_index()
    fed = fed.loc[~fed.index.duplicated(keep="last")]

    if start_date is not None:
        start_ts = pd.to_datetime(start_date)
        fed = fed[fed.index >= start_ts]
        print(f"[*] Filtre start_date appliqué: {start_ts.date()}")

    if end_date is not None:
        end_ts = pd.to_datetime(end_date)
        fed = fed[fed.index <= end_ts]
        print(f"[*] Filtre end_date appliqué: {end_ts.date()}")

    if fed.empty:
        print("[!] Aucune donnée Fed disponible après filtrage.")
        return fed

    fed = fed.asfreq("D").ffill()

    print(f"[✓] Couverture Fed: {fed.index.min().date()} -> {fed.index.max().date()}")
    print(f"[✓] Observations converties en quotidien: {len(fed)} lignes")

    print("\n" + "-"*70)
    print("STATISTIQUES TAUX D'INTÉRÊT (FED)")
    print("-"*70)
    print(f"Taux moyen: {fed['fed_real_rate'].mean():.3f}")
    print(f"Taux min:   {fed['fed_real_rate'].min():.3f}")
    print(f"Taux max:   {fed['fed_real_rate'].max():.3f}")

    return fed


# ============================================================================
# 4. FONCTION D'INTÉGRATION AU DATASET EXISTANT
# ============================================================================

def integrate_fear_greed(btc_data, fgi_data):
    """
    Intègre les données Fear & Greed au dataset Bitcoin existant.
    
    Args:
        btc_data (pd.DataFrame): Dataset Bitcoin avec index date
        fgi_data (pd.DataFrame): Dataset Fear & Greed avec index date
    
    Returns:
        pd.DataFrame: Dataset fusionné
    """
    print("\n" + "="*70)
    print("INTÉGRATION DES DONNÉES")
    print("="*70)
    
    print(f"[*] Lignes BTC avant fusion: {len(btc_data)}")
    print(f"[*] Lignes FGI disponibles: {len(fgi_data)}")
    
    # Fusion sur l'index date (left join pour garder toutes les dates BTC)
    merged = btc_data.join(fgi_data, how="left")
    
    print(f"[*] Lignes après fusion: {len(merged)}")
    
    # Vérifier les valeurs manquantes
    missing = merged["fgi_score"].isna().sum()
    if missing > 0:
        print(f"[!] {missing} valeurs FGI manquantes (dates avant historique FGI)")
        print("[*] Forward-fill appliqué pour combler les trous")
        merged["fgi_score"] = merged["fgi_score"].ffill()
        merged["fgi_classification"] = merged["fgi_classification"].ffill()
    
    print("[✓] Fusion complétée avec succès")
    
    return merged


# ============================================================================
# 5. FONCTION PRINCIPALE
# ============================================================================

def main():
    """
    Fonction principale qui orchestre le processus complet.
    """
    print("\n" + "="*70)
    print(" "*15 + "ENRICHISSEMENT DATASET BITCOIN")
    print(" "*7 + "Fear & Greed Index + Taux d'intérêt (Fed)")
    print("="*70)
    
    # --- Paramètres ---
    INPUT_FILE = "../data/multi_asset_dataset.csv"
    OUTPUT_FILE = "../data/bitcoin_with_fear_greed_fed.csv"
    OUTPUT_FILE_XLSX = OUTPUT_FILE.replace(".csv", ".xlsx")
    START_DATE = "2019-01-01"
    
    # --- 1. Chargement du dataset existant ---
    print("\n[1/6] CHARGEMENT DU DATASET EXISTANT")
    print("-"*70)
    
    try:
        btc = pd.read_csv(INPUT_FILE, parse_dates=["date"])
        btc = btc.set_index("date")
        print(f"[✓] Dataset chargé: {len(btc)} lignes")
        print(f"[*] Période: {btc.index.min()} à {btc.index.max()}")
        print(f"[*] Colonnes: {list(btc.columns)}")
    except Exception as e:
        print(f"[!] ERREUR lors du chargement: {e}")
        return
    
    # --- 2. Récupération Fear & Greed ---
    print("\n[2/6] RÉCUPÉRATION FEAR & GREED INDEX")
    print("-"*70)
    
    fgi = fetch_fear_greed(start_date=START_DATE)
    
    if fgi.empty:
        print("[!] Impossible de continuer sans données FGI")
        return
    
    # --- 3. Création des features ---
    print("\n[3/6] CRÉATION DES FEATURES")
    print("-"*70)
    
    fgi = create_fgi_features(fgi)
    
    # --- 4. Intégration Fear & Greed ---
    print("\n[4/6] INTÉGRATION FEAR & GREED AU DATASET")
    print("-"*70)
    
    merged = integrate_fear_greed(btc, fgi)
    
    # --- 5. Intégration des taux Fed ---
    print("\n[5/6] INTÉGRATION DES TAUX D'INTÉRÊT (FED)")
    print("-"*70)
    
    fed = load_fed_interest_rates(start_date=merged.index.min(), end_date=merged.index.max())
    
    if fed.empty:
        print("[!] Taux Fed non intégrés - aucune colonne 'fed_' ajoutée.")
    else:
        merged = merged.join(fed, how="left")
        missing_fed = merged["fed_real_rate"].isna().sum()
        if missing_fed > 0:
            print(f"[!] {missing_fed} valeurs Fed manquantes après fusion (dates hors couverture).")
            merged["fed_real_rate"] = merged["fed_real_rate"].ffill()
            print("[*] Forward-fill appliqué aux taux Fed.")
        else:
            print("[✓] Fusion des taux Fed réussie (aucune valeur manquante).")
    
    new_cols_fgi = [col for col in merged.columns if col.startswith("fgi_")]
    fed_cols = [col for col in merged.columns if col.startswith("fed_")]
    
    # --- 6. Sauvegarde ---
    print("\n[6/6] SAUVEGARDE")
    print("-"*70)
    
    try:
        merged_to_save = merged.reset_index()
        merged_to_save.to_csv(OUTPUT_FILE, index=False)
        merged_to_save.to_excel(OUTPUT_FILE_XLSX, index=False)
        print(f"[✓] Fichiers sauvegardés: {OUTPUT_FILE} et {OUTPUT_FILE_XLSX}")
        print(f"[*] {len(merged_to_save)} lignes")
        print(f"[*] {len(merged_to_save.columns)} colonnes")
        
        print(f"\n[*] Colonnes Fear & Greed ({len(new_cols_fgi)}):")
        for col in new_cols_fgi:
            print(f"    - {col}")
        
        if fed_cols:
            print(f"\n[*] Colonnes Fed ({len(fed_cols)}):")
            for col in fed_cols:
                print(f"    - {col}")
        else:
            print("\n[!] Aucune colonne Fed disponible dans le dataset final.")
        
    except Exception as e:
        print(f"[!] ERREUR lors de la sauvegarde: {e}")
        return
    
    # --- Statistiques finales ---
    print("\n" + "="*70)
    print("RÉSUMÉ FINAL")
    print("="*70)
    print(f"Dataset d'entrée:     {INPUT_FILE}")
    print(f"Dataset de sortie:    {OUTPUT_FILE}")
    print(f"Fichier Excel:        {OUTPUT_FILE_XLSX}")
    print(f"Lignes traitées:      {len(merged)}")
    print(f"Colonnes totales:     {len(merged.columns)}")
    print(f"Nouvelles colonnes:   {len(new_cols_fgi) + len(fed_cols)}")
    print(f"Période couverte:     {merged.index.min()} à {merged.index.max()}")
    
    if fed_cols:
        fed_col = fed_cols[0]
        coverage = merged[fed_col].notna().sum()
        print(f"Couverture taux Fed:  {coverage} jours ({coverage/len(merged)*100:.1f}%)")
        print(f"Dernier taux Fed:     {merged[fed_col].iloc[-1]:.3f}")
    
    # Distribution Fear & Greed
    print("\n" + "-"*70)
    print("DISTRIBUTION FEAR & GREED DANS LE DATASET FINAL")
    print("-"*70)
    fgi_dist = merged["fgi_category"].value_counts().sort_index()
    for cat, count in fgi_dist.items():
        pct = (count / len(merged)) * 100
        print(f"{cat:20s}: {count:5d} jours ({pct:5.1f}%)")
    
    print("\n" + "="*70)
    print("✓ PROCESSUS TERMINÉ AVEC SUCCÈS")
    print("="*70 + "\n")


# ============================================================================
# POINT D'ENTRÉE
# ============================================================================

if __name__ == "__main__":
    main()


               ENRICHISSEMENT DATASET BITCOIN
       Fear & Greed Index + Taux d'intérêt (Fed)

[1/6] CHARGEMENT DU DATASET EXISTANT
----------------------------------------------------------------------
[✓] Dataset chargé: 2497 lignes
[*] Période: 2019-01-01 00:00:00 à 2025-11-01 00:00:00
[*] Colonnes: ['close_BTC', 'ret_BTC', 'vol30_BTC', 'trend', 'vol_level', 'ret_bucket', 'month_name', 'weekday', 'close_ETH', 'vol_BTC', 'vol_ETH', 'close_SP500', 'close_GOLD', 'ret_ETH', 'ret_SP500', 'ret_GOLD', 'vol30_SP500', 'vol30_GOLD', 'vol30_ETH', 'corr30_BTC_SP500', 'corr30_BTC_GOLD', 'corr30_BTC_ETH', 'day', 'month']

[2/6] RÉCUPÉRATION FEAR & GREED INDEX
----------------------------------------------------------------------

RÉCUPÉRATION DE L'INDICE FEAR & GREED
[*] Requête vers: https://api.alternative.me/fng/?limit=0&format=json
[✓] 2833 enregistrements récupérés
[*] Période disponible: 2018-02-01 00:00:00 à 2025-11-07 00:00:00
[*] Période filtrée: 2019-01-01 00:00:00 à 2025-11-07 00:00: